# Hyperparameter search · binary

## Libraries and random seed

The seed identifies the run and the device determines where tensors are processed.

In [ ]:
import os
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# ── Run configuration ──
# The seed identifies the random choices in this run. A comparison across methods also
# depends on using the same training and test examples.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Hold out the test set

Each CSV row describes a flow. Swapping columns can preserve tensor shape while changing what each input means.

In [ ]:
df = pd.read_csv("Normal+Attacks.csv")

print(f"New dataset dimensions: {df.shape[0]} rows and {df.shape[1]} columns")

X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include="number")

TASK = "binary"

if TASK == "binary":
    y = df["label"].apply(lambda x: 0 if x.strip().lower() == "normal" else 1)
    n_classes = 2
else:
    le = LabelEncoder()
    y = le.fit_transform(df["label"])
    n_classes = len(le.classes_)

# ── Reference holdout ──
# Stratification aims to retain label proportions. Two rows from one capture can still land
# on opposite sides of this row-level split.
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

y_train = np.asarray(y_train_raw)

N_FEATURES = X_train_raw.shape[1]

print(f"\nData ready for scenario: {TASK.upper()}")
print(f"Number of numeric input columns (N_FEATURES): {N_FEATURES}")
print(f"Number of classes identified: {n_classes}")

## Network input shapes

B is batch size and F is feature count. The MLP receives [B, F]. Recurrent models traverse features as [B, F, 1]. The CNN receives [B, 1, F], with one channel. Here F indexes features of a flow.

In [ ]:

class MLP_Tuning(nn.Module):
    def __init__(self, input_dim, n_layers, units, dropouts, activation, n_classes):
        super().__init__()
        layers = []
        previous = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(previous, units[i]))
            layers.append(nn.ReLU() if activation == "relu" else nn.Tanh())
            if dropouts[i] > 0: layers.append(nn.Dropout(dropouts[i]))
            previous = units[i]
        layers.append(nn.Linear(previous, 1 if n_classes == 2 else n_classes))
        self.rede = nn.Sequential(*layers)
    def forward(self, x): return self.rede(x)

# ── Recurrent layers ──
# The traversed axis contains the features of one flow. Column order therefore defines the
# sequence seen by the network.
class Recorrente_Tuning(nn.Module):
    def __init__(self, architecture, input_dim, n_layers, units, dropouts, n_classes):
        super().__init__()
        self.architecture = architecture
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        previous = input_dim
        for i in range(n_layers):
            if architecture == "RNN": layer = nn.RNN(previous, units[i], batch_first=True)
            elif architecture == "GRU": layer = nn.GRU(previous, units[i], batch_first=True)
            else: layer = nn.LSTM(previous, units[i], batch_first=True)
            self.camadas.append(layer)
            self.dropouts.append(nn.Dropout(dropouts[i]) if dropouts[i] > 0 else nn.Identity())
            previous = units[i]
        self.densa = nn.Linear(previous, 64)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(64, 1 if n_classes == 2 else n_classes)
    def forward(self, x):
        for i, (layer, drop) in enumerate(zip(self.camadas, self.dropouts)):
            out, _ = layer(x)
            out = drop(out)
            if i < len(self.camadas) - 1: x = out
            else: x = out[:, -1, :] # Use only the last time step
        return self.saida(self.relu(self.densa(x)))

# ── Convolution over features ──
# The input has one channel. A filter combines neighboring features in CSV order, so column
# position matters.
class CNN_Tuning(nn.Module):
    def __init__(self, input_channels, n_blocos, filters, kernel_size, n_classes):
        super().__init__()
        layers = []
        previous = input_channels
        for i in range(n_blocos):
            layers.append(nn.Conv1d(previous, filters[i], kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool1d(kernel_size=2, ceil_mode=True))
            previous = filters[i]
        self.conv = nn.Sequential(*layers)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(previous, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.saida = nn.Linear(128, 1 if n_classes == 2 else n_classes)
    def forward(self, x):
        x = self.conv(x)
        x = self.pool_global(x).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))

## Model selection on validation data

Each Optuna trial trains one configuration and returns validation accuracy. Validation uses 10% of the training portion set aside for the search. The scaler learns from X_fit_raw and transforms X_val_raw using those parameters.

In [ ]:
DIR_TUNING = "bayesian_tuning"
os.makedirs(DIR_TUNING, exist_ok=True)

MAX_TRIALS = 25
EPOCHS_BUSCA = 25
BATCH_SIZE = 256

# ── Search validation ──
# Hyperparameter selection uses a reserved portion of the training data. Optuna trials can
# therefore be compared without consulting the test set.
X_fit_raw, X_val_raw, y_training, y_validation = train_test_split(
    X_train_raw, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)

scaler = StandardScaler()

# ── Scale fitted on training data ──
# Each column has its own mean and scale. The subsequent transform reuses these values even
# when the test distribution differs.
X_training = scaler.fit_transform(X_fit_raw)
X_validation = scaler.transform(X_val_raw)

train_dataset = TensorDataset(torch.tensor(X_training, dtype=torch.float32), torch.tensor(y_training, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(torch.tensor(X_validation, dtype=torch.float32), torch.tensor(y_validation, dtype=torch.float32))
val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False)

def criar_objetivo(model_name):

    # ── One Optuna trial ──
    # Each trial creates a new network. The accuracy returned to the study comes from
    # validation after the fitting epochs.
    def objetivo(trial):
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        
        if model_name == "MLP":
            n_layers = trial.suggest_int("n_camadas", 1, 3)
            units = [trial.suggest_int(f"unidades_{i}", 32, 256, step=32) for i in range(n_layers)]
            dropouts = [trial.suggest_float(f"dropout_{i}", 0.0, 0.5, step=0.1) for i in range(n_layers)]
            activation = trial.suggest_categorical("ativacao", ["relu", "tanh"])
            model = MLP_Tuning(N_FEATURES, n_layers, units, dropouts, activation, n_classes).to(DEVICE)
        
        elif model_name in ["RNN", "GRU", "LSTM"]:
            n_layers = trial.suggest_int("n_camadas", 1, 2)
            units = [trial.suggest_int(f"unidades_{i}", 16, 128, step=16) for i in range(n_layers)]
            dropouts = [trial.suggest_float(f"dropout_{i}", 0.0, 0.4, step=0.1) for i in range(n_layers)]
            model = Recorrente_Tuning(model_name, 1, n_layers, units, dropouts, n_classes).to(DEVICE)
            
        elif model_name == "CNN":
            n_blocos = trial.suggest_int("n_blocos_conv", 1, 3)
            filters = [trial.suggest_int(f"filtros_{i}", 16, 128, step=16) for i in range(n_blocos)]
            kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])
            model = CNN_Tuning(1, n_blocos, filters, kernel_size, n_classes).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(EPOCHS_BUSCA):
            model.train()
            for xb, y_batch in train_loader:
                xb, y_batch = xb.to(DEVICE), y_batch.to(DEVICE)
                if model_name in ["RNN", "GRU", "LSTM"]: xb = xb.unsqueeze(-1)
                elif model_name == "CNN": xb = xb.unsqueeze(-1).permute(0, 2, 1)
                
                optimizer.zero_grad()
                loss = criterion(model(xb).view(-1), y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        total_certos = 0
        total_samples = 0
        with torch.no_grad():
            for xb_v, yb_v in val_loader:
                xb_v, yb_v = xb_v.to(DEVICE), yb_v.to(DEVICE)
                if model_name in ["RNN", "GRU", "LSTM"]: xb_v = xb_v.unsqueeze(-1)
                elif model_name == "CNN": xb_v = xb_v.unsqueeze(-1).permute(0, 2, 1)
                
                preds = torch.sigmoid(model(xb_v)).view(-1)
                preds_binarias = (preds > 0.5).float()
                total_certos += (preds_binarias == yb_v).sum().item()
                total_samples += yb_v.size(0)
                
        return total_certos / total_samples

    return objetivo

MODEL_NAMES = ["MLP", "CNN", "RNN", "GRU", "LSTM"]

# ── Resuming a search ──
# Architectures in this list are read from existing JSON files. The remaining architectures
# enter the new trial queue.
COMPLETED_MODELS = ["MLP", "CNN", "RNN"]  # atualizar conforme forem terminando

overall_summary = {}
for model_name in COMPLETED_MODELS:
    path = os.path.join(
        DIR_TUNING, f"best_hyperparameters_{model_name}_{TASK}.json"
    )
    with open(path, "r") as f:
        overall_summary[model_name] = json.load(f)


for model_name in MODEL_NAMES:
    if model_name in overall_summary:
        continue

    print("\n" + "=" * 60)
    print(f"Starting hyperparameter tuning for {model_name} em {DEVICE}")
    print("=" * 60)
    
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(criar_objetivo(model_name), n_trials=MAX_TRIALS)
    
    print(f"Best accuracy for {model_name}: {study.best_value:.4f}")
    
    overall_summary[model_name] = {
        "val_accuracy": study.best_value,
        "hiperparametros": study.best_params
    }
    
    with open(os.path.join(DIR_TUNING, f"best_hyperparameters_{model_name}_{TASK}.json"), "w") as f:
        json.dump(overall_summary[model_name], f, indent=2)

summary_path = os.path.join(DIR_TUNING, f"resumo_best_hyperparameters_{TASK}.json")
with open(summary_path, "w") as f:
    json.dump(overall_summary, f, indent=2)

print("\n" + "=" * 60)
print(f"TUNING COMPLETED SUCCESSFULLY! FILE: {summary_path}")
print("=" * 60)